# Notebook 06: Synthetic Phi Recovery + Model Comparison
**Goal:**
1. Fix small-phi recovery using direct curve fitting.
2. Compare Standard QM vs shifted X-Theta phase model.
3. Show why CHSH-only phi recovery is weak near phi=0.
4. Demonstrate the X-Theta residual signature test.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path

from scipy.optimize import curve_fit
from scipy.stats import chi2

# Ensure output directory exists
output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

## 1. Theory functions

In [ ]:
def qm_standard(angle_diff):
    """
    Standard flat-space singlet correlation.
    """
    return -np.cos(angle_diff)


def xtheta_shifted(angle_diff, phi):
    """
    X-Theta shifted phase model.
    """
    return -np.cos(angle_diff + phi)


def xtheta_shifted_visibility(angle_diff, phi, visibility):
    """
    X-Theta model with phase shift and visibility/noise parameter.
    Useful for real experiments where contrast may be less than 1.
    """
    return -visibility * np.cos(angle_diff + phi)


def chsh_s_from_phi(phi):
    """
    For ideal optimal CHSH settings with common phase shift phi:
    |S(phi)| = 2 sqrt(2) |cos(phi)|
    """
    return 2 * np.sqrt(2) * abs(np.cos(phi))


def phi_from_chsh_s(S):
    """
    Invert ideal CHSH formula.
    WARNING:
    This is unstable near phi=0 because S changes quadratically.
    It also cannot recover the sign of phi.
    """
    ratio = np.clip(abs(S) / (2 * np.sqrt(2)), 0.0, 1.0)
    return np.arccos(ratio)

## 2. Synthetic data generation

In [ ]:
def make_chsh4_settings():
    """
    Four CHSH setting pairs.
    This mimics Hensen-style four-setting data.
    """
    return pd.DataFrame([
        {"setting": "00", "theta_A": 0.0,       "theta_B": np.pi / 4},
        {"setting": "01", "theta_A": 0.0,       "theta_B": -np.pi / 4},
        {"setting": "10", "theta_A": np.pi / 2, "theta_B": np.pi / 4},
        {"setting": "11", "theta_A": np.pi / 2, "theta_B": -np.pi / 4},
    ])


def make_angle_scan_settings(n_angles=41):
    """
    Many angle differences.
    This is much better for detecting small phase shifts.
    """
    angle_diffs = np.linspace(-np.pi, np.pi, n_angles)

    return pd.DataFrame({
        "setting": [f"scan_{i:02d}" for i in range(n_angles)],
        "theta_A": angle_diffs,
        "theta_B": np.zeros_like(angle_diffs),
    })


def simulate_binary_product_data(
    phi_true,
    settings_df,
    n_trials_per_setting=10000,
    visibility=1.0,
    seed=42
):
    """
    Generate synthetic Bell correlation data.

    Instead of directly adding Gaussian noise to E, this simulates binary
    product outcomes q = A*B in {-1,+1} with mean E.

    P(q=+1) = (1+E)/2
    P(q=-1) = (1-E)/2
    """

    rng = np.random.default_rng(seed)

    rows = []

    for row in settings_df.itertuples(index=False):
        angle_diff = row.theta_A - row.theta_B

        E_true = -visibility * np.cos(angle_diff + phi_true)

        p_plus = (1.0 + E_true) / 2.0
        p_plus = np.clip(p_plus, 0.0, 1.0)

        products = rng.choice(
            [+1, -1],
            size=n_trials_per_setting,
            p=[p_plus, 1.0 - p_plus]
        )

        E_hat = products.mean()
        N = len(products)

        # Standard error for mean of +/-1 variable
        error = np.sqrt(max(1.0 - E_hat**2, 1e-12) / N)

        rows.append({
            "setting": row.setting,
            "theta_A": row.theta_A,
            "theta_B": row.theta_B,
            "angle_diff": angle_diff,
            "N": N,
            "E_true": E_true,
            "E": E_hat,
            "Error": max(error, 1e-9),
            "phi_true": phi_true,
            "visibility_true": visibility,
        })

    return pd.DataFrame(rows)

## 3. CHSH calculation

In [ ]:
def compute_chsh_from_four_settings(df):
    """
    Compute CHSH from four setting rows: 00, 01, 10, 11.

    Uses:
    S = E00 + E01 + E10 - E11
    """

    lookup = {
        row.setting: row.E
        for row in df.itertuples(index=False)
    }

    err_lookup = {
        row.setting: row.Error
        for row in df.itertuples(index=False)
    }

    required = ["00", "01", "10", "11"]
    for key in required:
        if key not in lookup:
            raise ValueError(f"Missing setting {key}")

    S = lookup["00"] + lookup["01"] + lookup["10"] - lookup["11"]

    # Independent-error approximation
    S_error = np.sqrt(
        err_lookup["00"]**2 +
        err_lookup["01"]**2 +
        err_lookup["10"]**2 +
        err_lookup["11"]**2
    )

    return abs(S), S_error

## 4. Model fitting and comparison

In [ ]:
def weighted_chi2(y, y_pred, errors):
    errors = np.clip(errors, 1e-9, None)
    return float(np.sum(((y - y_pred) / errors) ** 2))


def aic_from_chi2(chi2_value, k):
    """
    Gaussian-error AIC up to an additive constant.
    Lower is better.
    """
    return chi2_value + 2 * k


def bic_from_chi2(chi2_value, k, n):
    """
    Gaussian-error BIC up to an additive constant.
    Lower is better.
    """
    return chi2_value + k * np.log(n)


def fit_models(df, fit_visibility=False):
    """
    Compare:

    Model 0: Standard QM
        E = -cos(angle_diff)

    Model 1: Shifted X-Theta
        E = -cos(angle_diff + phi)

    Optional Model 2:
        E = -V cos(angle_diff + phi)

    Returns a dictionary with fitted parameters and model comparison.
    """

    x = df["angle_diff"].values.astype(float)
    y = df["E"].values.astype(float)
    err = df["Error"].values.astype(float)
    n = len(y)

    # ----------------------------
    # Model 0: Standard QM, k=0
    # ----------------------------
    pred_qm = qm_standard(x)
    chi2_qm = weighted_chi2(y, pred_qm, err)
    rmse_qm = np.sqrt(np.mean((y - pred_qm)**2))

    result = {
        "n_points": n,
        "qm_chi2": chi2_qm,
        "qm_rmse": rmse_qm,
        "qm_k": 0,
        "qm_aic": aic_from_chi2(chi2_qm, 0),
        "qm_bic": bic_from_chi2(chi2_qm, 0, n),
    }

    # ----------------------------
    # Model 1: shifted phase, k=1
    # ----------------------------
    def model_shifted(x, phi):
        return xtheta_shifted(x, phi)

    popt, pcov = curve_fit(
        model_shifted,
        x,
        y,
        sigma=err,
        absolute_sigma=True,
        p0=[0.0],
        bounds=([-np.pi / 2], [np.pi / 2]),
        maxfev=10000
    )

    phi_hat = float(popt[0])
    phi_se = float(np.sqrt(np.diag(pcov))[0])

    pred_shifted = model_shifted(x, phi_hat)
    chi2_shifted = weighted_chi2(y, pred_shifted, err)
    rmse_shifted = np.sqrt(np.mean((y - pred_shifted)**2))

    delta_chi2 = chi2_qm - chi2_shifted

    # Since shifted model has one extra parameter, use chi-square survival
    # as an approximate likelihood-ratio p-value.
    p_value_lrt = float(chi2.sf(max(delta_chi2, 0.0), df=1))

    result.update({
        "phi_hat": phi_hat,
        "phi_se": phi_se,
        "phi_z": phi_hat / phi_se if phi_se > 0 else np.nan,
        "shifted_chi2": chi2_shifted,
        "shifted_rmse": rmse_shifted,
        "shifted_k": 1,
        "shifted_aic": aic_from_chi2(chi2_shifted, 1),
        "shifted_bic": bic_from_chi2(chi2_shifted, 1, n),
        "delta_chi2_qm_minus_shifted": delta_chi2,
        "lrt_p_value": p_value_lrt,
        "preferred_by_aic": "shifted_xtheta" if aic_from_chi2(chi2_shifted, 1) < aic_from_chi2(chi2_qm, 0) else "standard_qm",
        "preferred_by_bic": "shifted_xtheta" if bic_from_chi2(chi2_shifted, 1, n) < bic_from_chi2(chi2_qm, 0, n) else "standard_qm",
    })

    return result

## 5. Residual signature test

In [ ]:
def residual_signature_test(df):
    """
    Test whether residuals follow:
        Delta E ≈ phi * sin(angle_diff)

    This is the key X-Theta small-phase signature.
    """

    x = df["angle_diff"].values.astype(float)
    y = df["E"].values.astype(float)
    err = df["Error"].values.astype(float)

    qm_pred = qm_standard(x)
    residual = y - qm_pred

    signature = np.sin(x)

    # Estimate delta from linear regression:
    # residual ≈ phi * sin(x)
    weights = 1.0 / np.clip(err, 1e-9, None)**2

    numerator = np.sum(weights * signature * residual)
    denominator = np.sum(weights * signature**2)

    delta_signature = numerator / denominator
    delta_signature_se = np.sqrt(1.0 / denominator)

    z = delta_signature / delta_signature_se

    return {
        "delta_signature": delta_signature,
        "delta_signature_se": delta_signature_se,
        "z": z,
    }

## 6. Run recovery tests

We test for $\phi \in \{0.0, 0.005, 0.01, 0.02, 0.05, 0.1\}$.

In [ ]:
def run_recovery_experiment(
    mode,
    phi_values,
    n_trials_per_setting=10000,
    visibility=1.0,
    seed=42
):
    """
    mode:
    - "chsh4": only four CHSH settings
    - "angle_scan": many angles, stronger small-phi detection
    """

    if mode == "chsh4":
        settings = make_chsh4_settings()
    elif mode == "angle_scan":
        settings = make_angle_scan_settings(n_angles=41)
    else:
        raise ValueError("mode must be 'chsh4' or 'angle_scan'")

    rows = []

    for phi in phi_values:
        df = simulate_binary_product_data(
            phi_true=phi,
            settings_df=settings,
            n_trials_per_setting=n_trials_per_setting,
            visibility=visibility,
            seed=seed
        )

        fit = fit_models(df)
        sig = residual_signature_test(df)

        row = {
            "mode": mode,
            "phi_true": phi,
            "phi_hat_direct_fit": fit["phi_hat"],
            "phi_se": fit["phi_se"],
            "phi_z": fit["phi_z"],
            "qm_chi2": fit["qm_chi2"],
            "shifted_chi2": fit["shifted_chi2"],
            "delta_chi2": fit["delta_chi2_qm_minus_shifted"],
            "lrt_p_value": fit["lrt_p_value"],
            "preferred_by_aic": fit["preferred_by_aic"],
            "preferred_by_bic": fit["preferred_by_bic"],
            "signature_z": sig["z"],
            "qm_rmse": fit["qm_rmse"],
            "shifted_rmse": fit["shifted_rmse"],
        }

        if mode == "chsh4":
            S, S_error = compute_chsh_from_four_settings(df)
            row["CHSH_S"] = S
            row["CHSH_S_error" ] = S_error
            row["phi_from_chsh_only"] = phi_from_chsh_s(S)
        else:
            row["CHSH_S"] = np.nan
            row["CHSH_S_error"] = np.nan
            row["phi_from_chsh_only"] = np.nan

        rows.append(row)

    return pd.DataFrame(rows)


phi_values = [0.0, 0.005, 0.01, 0.02, 0.05, 0.1]

df_chsh4 = run_recovery_experiment(
    mode="chsh4",
    phi_values=phi_values,
    n_trials_per_setting=100000, # Large N to see small effects
    visibility=1.0,
    seed=42
)

df_scan = run_recovery_experiment(
    mode="angle_scan",
    phi_values=phi_values,
    n_trials_per_setting=10000,
    visibility=1.0,
    seed=42
)

print("CHSH-4 recovery:")
display(df_chsh4)

print("Angle-scan recovery:")
display(df_scan)

## 7. Visualization

In [ ]:
# 7.1 Recovery Accuracy
plt.figure(figsize=(8, 6))
plt.errorbar(df_chsh4["phi_true"], df_chsh4["phi_hat_direct_fit"], yerr=df_chsh4["phi_se"], fmt="o-", label="4 CHSH settings")
plt.errorbar(df_scan["phi_true"], df_scan["phi_hat_direct_fit"], yerr=df_scan["phi_se"], fmt="s-", label="Angle scan")
plt.plot([0, 0.1], [0, 0.1], "k--", label="Ideal")
plt.xlabel("True $\\phi$")
plt.ylabel("Recovered $\\hat{\\phi}$")
plt.title("Synthetic Phi Recovery Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(output_dir / "synthetic_phi_recovery_model_fit.png")
plt.show()

# 7.2 CHSH Instability
plt.figure(figsize=(8, 6))
plt.plot(df_chsh4["phi_true"], df_chsh4["phi_from_chsh_only"], "o-", label="Phi from CHSH magnitude")
plt.plot([0, 0.1], [0, 0.1], "k--", label="Ideal")
plt.xlabel("True $\\phi$")
plt.ylabel("Recovered $\\phi$ (CHSH-only)")
plt.title("CHSH-Only Phi Recovery Is Unstable Near Zero")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(output_dir / "synthetic_phi_recovery_small_phi_error.png")
plt.show()

# 7.3 Model Comparison (Delta Chi2)
plt.figure(figsize=(8, 6))
plt.plot(df_chsh4["phi_true"], df_chsh4["delta_chi2"], "o-", label="4 CHSH settings")
plt.plot(df_scan["phi_true"], df_scan["delta_chi2"], "s-", label="Angle scan")
plt.axhline(3.84, color="r", linestyle="--", label="95% Confidence Threshold")
plt.xlabel("True $\\phi$")
plt.ylabel("$\\Delta\\chi^2 = \\chi^2_{QM} - \\chi^2_{shifted}$")
plt.title("Model Comparison: X-Theta Improvement over QM")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(output_dir / "synthetic_phi_recovery_model_comparison.png")
plt.show()

# 7.4 Residual Signature (Z-score)
plt.figure(figsize=(8, 6))
plt.plot(df_scan["phi_true"], df_scan["signature_z"], "s-", label="Residual Signature Z-score")
plt.axhline(2.0, color="r", linestyle="--", label="Z=2 threshold")
plt.xlabel("True $\\phi$")
plt.ylabel("Signature Z-score")
plt.title("Falsifiable X-Theta Residual Signature")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(output_dir / "synthetic_phi_recovery_residuals.png")
plt.show()

## Milestone 4: Unique Physical Prediction

Standard QM can fit the dominant correlation curve, but it should not recover a stable hidden phase-offset parameter from residual structure. The shifted X-Theta phase model predicts a small but recoverable phase signature, especially visible through residual asymmetry and improved small-phi recovery.

> **Crucial Claim:** Unlike standard flat-space quantum mechanics, which predicts no residual path-dependent angular phase after calibration, X-Theta predicts a transport-induced phase holonomy ($\delta_{\gamma}$) whose observable signature is a first-order residual ($\Delta E \approx \delta_{\gamma} \sin(\theta_a - \theta_b)$).

**Caveat:** This is a synthetic falsification/recovery test. It does not prove the theory from real experimental data, but it defines a measurable prediction that can later be tested against open Bell datasets.